In [17]:
from dataclasses import dataclass
import os
from typing import Optional
from openai import OpenAI
@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests across all inference providers"""
    name:str
    env_var:str
    is_free:bool
    base_url: Optional[str]
    model:str

PROVIDERS = [
   Provider("Groq","GROQ_API_KEY",True,"https://api.groq.com/openai/v1","openai/gpt-oss-120b"),
   Provider("OpenAI","OPENAI_API_KEY",True,None,"gpt-4o-mini"),    
]

def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:


    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )


def have_any_key()->bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")    



Found a provider key.


In [18]:
def llm_reply(prompt:str)->str:
    provider = select_provider()
    print(provider)
    client = build_client(provider)
    result = client.chat.completions.create(
        model = provider.model,
        max_tokens=200,
        messages=[
            {
                "role":"user",
                "content":prompt,
            }
        ]
    )

    return result.choices[0].message.content

In [19]:
prompt = "Who was the PM of UK before 2019? Answer in single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error:{e}")

Provider(name='Groq', env_var='GROQ_API_KEY', is_free=True, base_url='https://api.groq.com/openai/v1', model='openai/gpt-oss-120b')
Theresa May served as the United Kingdom’s Prime Minister from July 2016 until July 2019.


In [24]:
from typing import List,Dict

def chat_reply(messages:List[Dict])-> str:
    provider= select_provider()

    client = build_client(provider)
    
    result = client.chat.completions.create(
        model = provider.model,
        max_tokens = 1000,
        messages = messages,
    )

    return result.choices[0].message.content

In [25]:
conversation = []

conversation.append(
    {
        "role":"assistant",
        "content":"What is the capital of France?",
    }
)

In [26]:
reply_from_llm = chat_reply(conversation)
print(reply_from_llm)
conversation.append({
    "role":"assistant",
    "content":reply_from_llm,
})

The capital of France is **Paris**.


In [27]:
conversation

[{'role': 'assistant', 'content': 'What is the capital of France?'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'}]

In [ ]:
conversation.append({
    "role":"user",
    "content":"what is there gdp"
})

In [ ]:
llm_reply = chat_reply(conversation)
print(llm_reply)

In [29]:
conversation

[{'role': 'assistant', 'content': 'What is the capital of France?'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'},
 {'role': 'user', 'content': 'what is there gdp'}]